## Step 0 Env, Constants and Definitions

In [1]:
MODEL_NAME = "meta-llama/Llama-3.2-1B"
TEXT_COLUMN = "title"
LABEL_COLUMN = "main_category"
ID_COLUMN = "asin"

RANDOM_SEED = 30
TEST_SIZE = 0.10
VALIDATION_SIZE = 0.10

print(f"Base model: {MODEL_NAME}")
print(f"Input column: {TEXT_COLUMN}")
print(f"Label column: {LABEL_COLUMN}")

Base model: meta-llama/Llama-3.2-1B
Input column: title
Label column: main_category


In [2]:
from __future__ import annotations

import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import psycopg
import torch
from dotenv import load_dotenv
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from huggingface_hub import login, whoami

load_dotenv()

OUTPUT_DIR = Path("outputs/llama3.1_1B")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = '../../asset/data/amazon_products_with_main_category.csv'
LABEL_MAP_PATH = OUTPUT_DIR / "label_map.json"
METRICS_PATH = OUTPUT_DIR / "metrics.json"
CONFUSION_MATRIX_CSV_PATH = OUTPUT_DIR / "confusion_matrix.csv"

np.random.seed(RANDOM_SEED)

### Step 0.1 Hugging Face Auth

In [3]:
HF_TOKEN = os.getenv("HF_TOKEN")

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    user_info = whoami(token=HF_TOKEN)
    print(f"Authenticated with Hugging Face as: {user_info['name']}")
else:
    print("HF_TOKEN is not set.")
    print("Set HF_TOKEN in your environment/.env, or run `huggingface-cli login` in a terminal before using the Gemma model.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Authenticated with Hugging Face as: yuechenzhang


## Step 1. Data Prep

### Step 1.1 Data Loading

In [4]:
LIMIT_ROWS: int | None = 100_000

df = pd.read_csv(
    CSV_PATH,
    usecols=["asin", "title", "main_category"],
)

print(f"Rows in amazon_products CSV: {len(df):,}")

# Remove missing values before string conversion
df = df.dropna(
    subset=[TEXT_COLUMN, LABEL_COLUMN]
)

# Clean text and label columns
df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str).str.strip()
df[LABEL_COLUMN] = df[LABEL_COLUMN].astype(str).str.strip()

df = (
    df[
        (df[TEXT_COLUMN] != "")
        & (df[LABEL_COLUMN] != "")
    ]
    .reset_index(drop=True)
)

# Deterministically sample rows after cleaning
if LIMIT_ROWS is not None and len(df) > LIMIT_ROWS:
    df = (
        df.sample(
            n=LIMIT_ROWS,
            random_state=RANDOM_SEED,
            replace=False,
        )
        .reset_index(drop=True)
    )

print(f"Rows used after cleaning and sampling: {len(df):,}")

df.head()

Rows in amazon_products CSV: 1,426,337
Rows used after cleaning and sampling: 100,000


,asin,title,main_category
0,B0BN1PTHGX,"Premium Jewelry Stand , Solid Clear 2-Tier Acr...",Home & Kitchen
1,B0C3TSRMBC,HPA200 Replacement Filters for Honeywell HPA20...,Home & Kitchen
2,B0CB88NJDN,"Couch Side Table with Adjustable Heights, Bamb...",Home & Kitchen
3,B089FKMQJR,Ironing Board Cover and Pad Extra Thick Heavy ...,Home & Kitchen
4,B082YS2K52,for Raspberry Pi 4 Aluminum Case with Fan and ...,Electronics & Computers


### Step 1.2 label Encoding

In [5]:
labels = sorted(df[LABEL_COLUMN].unique().tolist())
label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for label, idx in label2id.items()}

df["label"] = df[LABEL_COLUMN].map(label2id)

label_map = {
    "label2id": label2id,
    "id2label": {str(idx): label for idx, label in id2label.items()},
}
LABEL_MAP_PATH.write_text(json.dumps(label_map, indent=2), encoding="utf-8")

print(f"Saved label map to: {LABEL_MAP_PATH}")
label_map

Saved label map to: outputs/llama3.1_1B/label_map.json


{'label2id': {'Arts, Crafts & Party Supplies': 0,
  'Automotive': 1,
  'Baby Products': 2,
  'Beauty & Personal Care': 3,
  'Electronics & Computers': 4,
  'Fashion, Shoes & Luggage': 5,
  'Gift Cards': 6,
  'Health & Household': 7,
  'Home & Kitchen': 8,
  'Industrial & Scientific': 9,
  'Pet Supplies': 10,
  'Smart Home': 11,
  'Sports & Outdoors': 12,
  'Tools & Home Improvement': 13,
  'Toys & Games': 14,
  'Video Games': 15},
 'id2label': {'0': 'Arts, Crafts & Party Supplies',
  '1': 'Automotive',
  '2': 'Baby Products',
  '3': 'Beauty & Personal Care',
  '4': 'Electronics & Computers',
  '5': 'Fashion, Shoes & Luggage',
  '6': 'Gift Cards',
  '7': 'Health & Household',
  '8': 'Home & Kitchen',
  '9': 'Industrial & Scientific',
  '10': 'Pet Supplies',
  '11': 'Smart Home',
  '12': 'Sports & Outdoors',
  '13': 'Tools & Home Improvement',
  '14': 'Toys & Games',
  '15': 'Video Games'}}

### Step 1.3 Train Test Split

In [6]:
train_val_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=df["label"],
)

validation_fraction_of_train_val = VALIDATION_SIZE / (1.0 - TEST_SIZE)
train_df, val_df = train_test_split(
    train_val_df,
    test_size=validation_fraction_of_train_val,
    random_state=RANDOM_SEED,
    stratify=train_val_df["label"],
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train rows: {len(train_df):,}")
print(f"Validation rows: {len(val_df):,}")
print(f"Test rows: {len(test_df):,}")

Train rows: 80,000
Validation rows: 10,000
Test rows: 10,000


### Step 1.4 Convert Pandas DataFrames to Hugging Face Datasets

In [7]:
from datasets import Dataset, DatasetDict

model_columns = [ID_COLUMN, TEXT_COLUMN, LABEL_COLUMN, "label"]
dataset = DatasetDict(
    {
        "train": Dataset.from_pandas(train_df[model_columns], preserve_index=False),
        "validation": Dataset.from_pandas(val_df[model_columns], preserve_index=False),
        "test": Dataset.from_pandas(test_df[model_columns], preserve_index=False),
    }
)

print("sample_weight" in dataset["train"].column_names)
dataset

False


DatasetDict({
    train: Dataset({
        features: ['asin', 'title', 'main_category', 'label'],
        num_rows: 80000
    })
    validation: Dataset({
        features: ['asin', 'title', 'main_category', 'label'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['asin', 'title', 'main_category', 'label'],
        num_rows: 10000
    })
})

## Step 2 Baseline Setup with Prompt Only Classification

In [8]:
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

PROMPT_ONLY_EVAL_ROWS: int | None = 1000
PROMPT_UNKNOWN_LABEL = "__UNKNOWN__"
PROMPT_ONLY_METRICS_PATH = OUTPUT_DIR / "prompt_only_metrics.json"
PROMPT_ONLY_CONFUSION_MATRIX_CSV_PATH = OUTPUT_DIR / "prompt_only_confusion_matrix.csv"
PROMPT_ONLY_PREDICTIONS_CSV_PATH = OUTPUT_DIR / "prompt_only_predictions.csv"

def make_prompt_baseline_sample(source_df: pd.DataFrame, n: int | None) -> pd.DataFrame:
    if n is None or n >= len(source_df):
        return source_df.copy().reset_index(drop=True)
    return source_df.sample(n=n, random_state=RANDOM_SEED).reset_index(drop=True)

prompt_baseline_df = make_prompt_baseline_sample(test_df, PROMPT_ONLY_EVAL_ROWS)

print(f"Prompt-only evaluation rows: {len(prompt_baseline_df):,}")
display(prompt_baseline_df[LABEL_COLUMN].value_counts().rename("count").to_frame())

Prompt-only evaluation rows: 1,000


,count
main_category,
"Fashion, Shoes & Luggage",213
Industrial & Scientific,115
Toys & Games,102
Electronics & Computers,91
Home & Kitchen,81
Automotive,71
"Arts, Crafts & Party Supplies",59
Baby Products,55
Tools & Home Improvement,53


In [9]:
prompt_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, token=HF_TOKEN)
if prompt_tokenizer.pad_token is None:
    prompt_tokenizer.pad_token = prompt_tokenizer.eos_token

prompt_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    token=HF_TOKEN,
)
prompt_model.eval()

label_options = "\n".join(f"- {label}" for label in labels)

def build_prompt_only_category_prompt(title: str) -> str:
    return f"""
You are classifying product titles given product description.
Choose exactly one main_category from the allowed categories.
Return only the category name and no extra text.

Allowed categories:
{label_options}

Product title: {title}

main_category:
"""

def parse_prompt_category(generated_text: str) -> str | None:
    cleaned = generated_text.strip()
    first_line = cleaned.splitlines()[0].strip() if cleaned else ""
    first_line = first_line.strip("`'\" ,.:;{}[]()")

    label_lookup = {label.lower(): label for label in labels}
    if first_line.lower() in label_lookup:
        return label_lookup[first_line.lower()]

    lowered = cleaned.lower()
    matches = [label for label in labels if label.lower() in lowered]
    if len(matches) == 1:
        return matches[0]
    return None

def predict_prompt_only_main_category(title: str, max_new_tokens: int = 32) -> dict[str, str | None]:
    prompt = build_prompt_only_category_prompt(title)
    encoded = prompt_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    encoded = {key: value.to(prompt_model.device) for key, value in encoded.items()}

    with torch.no_grad():
        output_ids = prompt_model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=prompt_tokenizer.pad_token_id,
        )

    generated_ids = output_ids[0, encoded["input_ids"].shape[-1]:]
    generated_text = prompt_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    return {
        "generated_text": generated_text,
        "predicted_label": parse_prompt_category(generated_text),
    }

# Quick sanity check before running the full prompt-only sample.
predict_prompt_only_main_category("Wireless Bluetooth Noise Cancelling Headphones with Microphone")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


{'generated_text': '- Electronics & Computers\n- Home & Kitchen\n- Smart Home\n- Tools & Home Improvement\n- Toys & Games\n- Video Games',
 'predicted_label': None}

In [10]:
prompt_predictions = []
for row in tqdm(prompt_baseline_df.itertuples(index=False), total=len(prompt_baseline_df), desc="Prompt-only eval"):
    result = predict_prompt_only_main_category(getattr(row, TEXT_COLUMN))
    prompt_predictions.append(result)

prompt_eval_results = prompt_baseline_df[[ID_COLUMN, TEXT_COLUMN, LABEL_COLUMN, "label"]].copy()
prompt_eval_results["generated_text"] = [item["generated_text"] for item in prompt_predictions]
prompt_eval_results["predicted_main_category"] = [
    item["predicted_label"] if item["predicted_label"] is not None else PROMPT_UNKNOWN_LABEL
    for item in prompt_predictions
]
prompt_eval_results["predicted_label"] = prompt_eval_results["predicted_main_category"].map(label2id).fillna(-1).astype(int)

prompt_y_true = prompt_eval_results["label"].to_numpy()
prompt_y_pred = prompt_eval_results["predicted_label"].to_numpy()

prompt_only_metrics = {
    "eval_rows": int(len(prompt_eval_results)),
    "parse_success_rate": float((prompt_eval_results["predicted_main_category"] != PROMPT_UNKNOWN_LABEL).mean()),
    "accuracy": float(accuracy_score(prompt_y_true, prompt_y_pred)),
    "macro_f1": float(f1_score(prompt_y_true, prompt_y_pred, labels=list(range(len(labels))), average="macro", zero_division=0)),
}

prompt_cm_df = pd.crosstab(
    pd.Categorical(prompt_eval_results[LABEL_COLUMN], categories=labels),
    pd.Categorical(prompt_eval_results["predicted_main_category"], categories=labels + [PROMPT_UNKNOWN_LABEL]),
    rownames=["true_main_category"],
    colnames=["predicted_main_category"],
    dropna=False,
)

PROMPT_ONLY_METRICS_PATH.write_text(json.dumps(prompt_only_metrics, indent=2), encoding="utf-8")
prompt_cm_df.to_csv(PROMPT_ONLY_CONFUSION_MATRIX_CSV_PATH)
prompt_eval_results.to_csv(PROMPT_ONLY_PREDICTIONS_CSV_PATH, index=False)

print(prompt_only_metrics)
display(prompt_cm_df)
display(prompt_eval_results.head(20))

Prompt-only eval:   0%|          | 0/1000 [00:00<?, ?it/s]

{'eval_rows': 1000, 'parse_success_rate': 0.376, 'accuracy': 0.099, 'macro_f1': 0.08285723211215021}


predicted_main_category,"Arts, Crafts & Party Supplies",Automotive,Baby Products,Beauty & Personal Care,Electronics & Computers,"Fashion, Shoes & Luggage",Gift Cards,Health & Household,Home & Kitchen,Industrial & Scientific,Pet Supplies,Smart Home,Sports & Outdoors,Tools & Home Improvement,Toys & Games,Video Games,__UNKNOWN__
true_main_category,,,,,,,,,,,,,,,,,
"Arts, Crafts & Party Supplies",5,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,49
Automotive,0,57,0,0,0,0,0,0,0,0,0,0,0,0,0,0,14
Baby Products,0,2,32,0,0,0,0,0,0,0,0,0,1,0,0,0,20
Beauty & Personal Care,0,5,0,4,0,0,0,0,0,0,0,0,0,0,0,0,21
Electronics & Computers,0,29,0,0,0,0,0,0,0,0,0,0,0,0,0,0,62
"Fashion, Shoes & Luggage",2,53,4,1,0,1,0,0,0,0,0,0,2,0,0,0,150
Gift Cards,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Health & Household,0,8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,29
Home & Kitchen,0,17,2,1,0,0,0,0,0,0,0,0,0,0,0,0,61


,asin,title,main_category,label,generated_text,predicted_main_category,predicted_label
0,B0C6SC18K7,Toddlers Suction Bath Travel Toys - 24 Pack Ki...,Toys & Games,14,- Baby Products\n- Beauty & Personal Care\n- E...,__UNKNOWN__,-1
1,B09SVKNY9W,"Kelso Men’s Zero Drop Tennis Shoes, Lightweigh...","Fashion, Shoes & Luggage",5,- Automotive,Automotive,1
2,B07P13JTGB,Greenery Eucalyptus Baby Shower Bingo Cards - ...,Home & Kitchen,8,"- Arts, Crafts & Party Supplies\n- Automotive\...",__UNKNOWN__,-1
3,B09NWDXVS4,"Special Supplies Stepping Stones for Kids, 5 B...",Toys & Games,14,- Automotive\n- Baby Products\n- Beauty & Pers...,__UNKNOWN__,-1
4,B07S3TMXNT,3 Layering Necklace for Women 14K Gold Plated ...,"Fashion, Shoes & Luggage",5,- Automotive\n- Baby Products\n- Beauty & Pers...,__UNKNOWN__,-1
5,B00NGGG2TI,Paula's Choice RESIST Weightless Advanced Repa...,Beauty & Personal Care,3,- Beauty & Personal Care\n- Electronics & Comp...,__UNKNOWN__,-1
6,B00Y35HCGS,"Oreck Belt, Upright Flat (Pack of 3)",Home & Kitchen,8,- Automotive\n- Baby Products\n- Beauty & Pers...,__UNKNOWN__,-1
7,B08Y914T5G,HyDren 8 Pieces Plastic Sand Shovels 8.5 Inch ...,Toys & Games,14,"- Arts, Crafts & Party Supplies\n- Automotive\...",__UNKNOWN__,-1
8,B01NAAKRPK,Mozlly Monster Truck Toys Set with Trailer Toy...,Toys & Games,14,- Automotive,Automotive,1
9,B09Z9MV5KR,Hoodie for Boys Fleece Jacket Zip Up Sherpa Li...,"Fashion, Shoes & Luggage",5,- Baby Products,Baby Products,2


In [11]:
# Run this after prompt-only evaluation if you plan to continue with fine-tuning in the same kernel.
del prompt_model
# Keep prompt_tokenizer only if you want to inspect prompt-only outputs later.
del prompt_tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Released prompt-only model memory.")

Released prompt-only model memory.


## Step 3 LoRA Fine-Tuning

### Step 3.1 Tokenization

In [12]:
from transformers import AutoTokenizer

MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_batch(batch):
    return tokenizer(
        batch[TEXT_COLUMN],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

tokenized_dataset = dataset.map(tokenize_batch, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns([ID_COLUMN, TEXT_COLUMN, LABEL_COLUMN])
tokenized_dataset.set_format("torch")

tokenized_dataset

Map:   0%|          | 0/80000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 80000
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 10000
    })
})

### Step 3.2 LoRA Configs

In [13]:
import torch
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForSequenceClassification

num_labels = len(labels)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    token=HF_TOKEN,
)
model.config.pad_token_id = tokenizer.pad_token_id

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=10,
    lora_alpha=20,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

[transformers] LlamaForSequenceClassification LOAD REPORT from: meta-llama/Llama-3.2-1B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 7,077,888 || all params: 1,242,925,056 || trainable%: 0.5695


## Step 3.3 Training

In [14]:
def compute_metrics(eval_pred):
    logits, true_labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(true_labels, predictions),
        "macro_f1": f1_score(true_labels, predictions, average="macro"),
    }

from transformers import DataCollatorWithPadding, Trainer, TrainingArguments

TRAINING_OUTPUT_DIR = OUTPUT_DIR / "trainer"

training_args = TrainingArguments(
    output_dir=str(TRAINING_OUTPUT_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none",
    seed=RANDOM_SEED,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
trainer.save_model(str(OUTPUT_DIR / "adapter_model"))
tokenizer.save_pretrained(str(OUTPUT_DIR / "tokenizer"))
train_result

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.373346,0.367187,0.881300,0.830543


TrainOutput(global_step=2500, training_loss=1.8914913940429687, metrics={'train_runtime': 1091.0819, 'train_samples_per_second': 73.322, 'train_steps_per_second': 2.291, 'total_flos': 6.022697582592e+16, 'train_loss': 1.8914913940429687, 'epoch': 1.0})

### Step 3.4 Evaluations

In [15]:
test_predictions = trainer.predict(tokenized_dataset["test"])
y_true = test_predictions.label_ids
y_pred = np.argmax(test_predictions.predictions, axis=-1)

metrics = {
    "accuracy": accuracy_score(y_true, y_pred),
    "macro_f1": f1_score(y_true, y_pred, average="macro"),
}
METRICS_PATH.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

cm = confusion_matrix(y_true, y_pred, labels=list(range(num_labels)))
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
cm_df.to_csv(CONFUSION_MATRIX_CSV_PATH)

print(metrics)
display(cm_df)

{'accuracy': 0.8767, 'macro_f1': 0.811902617695258}


,"Arts, Crafts & Party Supplies",Automotive,Baby Products,Beauty & Personal Care,Electronics & Computers,"Fashion, Shoes & Luggage",Gift Cards,Health & Household,Home & Kitchen,Industrial & Scientific,Pet Supplies,Smart Home,Sports & Outdoors,Tools & Home Improvement,Toys & Games,Video Games
"Arts, Crafts & Party Supplies",649,0,0,2,9,11,0,1,19,22,1,0,0,3,13,1
Automotive,4,646,3,2,26,5,0,1,8,25,1,0,3,19,8,1
Baby Products,4,5,410,2,0,33,0,2,21,4,3,0,0,3,30,1
Beauty & Personal Care,1,0,0,335,0,11,0,20,5,3,1,0,0,0,0,0
Electronics & Computers,0,17,1,0,976,6,0,1,5,5,0,1,0,12,4,9
"Fashion, Shoes & Luggage",6,6,18,0,7,2100,0,3,7,4,1,0,3,3,4,0
Gift Cards,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
Health & Household,1,3,1,15,3,5,0,236,10,17,2,0,1,5,0,0
Home & Kitchen,32,8,25,2,9,10,0,9,744,37,2,0,2,20,25,1
Industrial & Scientific,22,22,6,3,6,9,0,26,32,900,3,0,2,42,1,0


## Step 4 Inference

In [16]:
def predict_main_category(title: str) -> dict[str, object]:
    encoded = tokenizer(
        title,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
    )
    encoded = {key: value.to(model.device) for key, value in encoded.items()}

    model.eval()
    with torch.no_grad():
        logits = model(**encoded).logits.float()
        probabilities = torch.softmax(logits, dim=-1).detach().cpu().numpy()[0]

    predicted_id = int(np.argmax(probabilities))
    return {
        "title": title,
        "predicted_label": id2label[predicted_id],
        "confidence": float(probabilities[predicted_id]),
    }

# Example after training:
predict_main_category("Wireless Bluetooth Noise Cancelling Headphones with Microphone")

{'title': 'Wireless Bluetooth Noise Cancelling Headphones with Microphone',
 'predicted_label': 'Electronics & Computers',
 'confidence': 0.9921956062316895}

## Step 5. Baseline vs Fine-Tuned Model Comparison

In [17]:
PROMPT_ONLY_COMPARISON_LABEL = "prompt_only_base_model"
FINE_TUNED_COMPARISON_LABEL = "lora_fine_tuned_classifier"
COMPARISON_METRICS_PATH = OUTPUT_DIR / "prompt_only_vs_fine_tuned_metrics.csv"
COMPARISON_RECALL_PATH = OUTPUT_DIR / "prompt_only_vs_fine_tuned_recall_by_class.csv"

if "prompt_only_metrics" not in globals():
    prompt_only_metrics = json.loads(PROMPT_ONLY_METRICS_PATH.read_text(encoding="utf-8"))

if "metrics" not in globals():
    metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))

comparison_df = pd.DataFrame(
    [
        {
            "run": PROMPT_ONLY_COMPARISON_LABEL,
            "eval_rows": prompt_only_metrics.get("eval_rows", PROMPT_ONLY_EVAL_ROWS),
            "accuracy": prompt_only_metrics["accuracy"],
            "macro_f1": prompt_only_metrics["macro_f1"],
        },
        {
            "run": FINE_TUNED_COMPARISON_LABEL,
            "eval_rows": len(test_df),
            "accuracy": metrics["accuracy"],
            "macro_f1": metrics["macro_f1"],
        },
    ]
)

fine_tuned_row = comparison_df.loc[comparison_df["run"] == FINE_TUNED_COMPARISON_LABEL].iloc[0]
prompt_only_row = comparison_df.loc[comparison_df["run"] == PROMPT_ONLY_COMPARISON_LABEL].iloc[0]

improvement_df = pd.DataFrame(
    [
        {
            "metric": "accuracy",
            "prompt_only": prompt_only_row["accuracy"],
            "fine_tuned": fine_tuned_row["accuracy"],
            "absolute_improvement": fine_tuned_row["accuracy"] - prompt_only_row["accuracy"],
        },
        {
            "metric": "macro_f1",
            "prompt_only": prompt_only_row["macro_f1"],
            "fine_tuned": fine_tuned_row["macro_f1"],
            "absolute_improvement": fine_tuned_row["macro_f1"] - prompt_only_row["macro_f1"],
        },
    ]
)

comparison_df.to_csv(COMPARISON_METRICS_PATH, index=False)

print("Overall comparison")
display(comparison_df)
print("Fine-tuning improvement")
display(improvement_df)

Overall comparison


,run,eval_rows,accuracy,macro_f1
0,prompt_only_base_model,1000,0.0990,0.082857
1,lora_fine_tuned_classifier,10000,0.8767,0.811903


Fine-tuning improvement


,metric,prompt_only,fine_tuned,absolute_improvement
0,accuracy,0.099000,0.876700,0.777700
1,macro_f1,0.082857,0.811903,0.729045
